# 01-2 Token 为什么能被计算？

在 [上一节](./01-1-token_what_and_where.ipynb) 中，我们搞清楚了 Token 是什么、从哪里来。但切分完的 Token 还是离散的整数 ID（如 `95933`），神经网络无法直接计算。

本节我们将揭示：**Token ID 如何变成可计算的稠密向量？这些向量真的能表达语义吗？**

核心概念是 **Embedding（嵌入）**——把离散的 Token ID 映射为连续的稠密向量。这些向量是跟模型一起训练出来的，语义相近的 Token 在高维空间中应该更接近。
让我们从三个角度验证：

## 1. 向量化：Token 如何变成可计算的？

切分完的 Token 还是离散的整数 ID（如 `95933`），神经网络无法直接计算。
我们需要把它们映射为连续的稠密向量——这就是 **Embedding**。

In [1]:
from modelscope import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("modelscope").setLevel(logging.ERROR)

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False

tokenizer_id = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)
model = AutoModelForCausalLM.from_pretrained(tokenizer_id)
model.eval()

embedding_layer = model.get_input_embeddings()
print(f"Embedding 层结构: {embedding_layer}")

print(f"\nEmbedding 维度: {embedding_layer.embedding_dim}")
print(f"词表大小: {embedding_layer.num_embeddings}")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Embedding 层结构: Embedding(248320, 1024)

Embedding 维度: 1024
词表大小: 248320


### 1.1 语义相似度：相近的词，向量真的更近吗？

如果 Embedding 学得好，语义相近的词（如"猫"和"狗"）的向量应该比语义无关的词（如"猫"和"桌子"）更接近。
我们用**余弦相似度**来衡量两个向量的接近程度（范围 -1 到 1，越大越相似）：

In [5]:
print("=== 语义相似度对比 ===\n")

similarity_groups = [
    ("同类对比", [
        ("king", "queen"),
        ("king", "man"),
        ("king", "apple"),
    ]),
    ("中文同类对比", [
        ("猫", "狗"),
        ("猫", "桌子"),
        ("苹果", "香蕉"),
        ("苹果", "汽车"),
    ]),
]

def get_embedding(word):
    ids = tokenizer.encode(word)
    emb = embedding_layer(torch.tensor([ids[0]])).detach()
    result = emb.squeeze(0)
    if word == '猫':
        print(f'get_embedding 维度细节（输入词：{word}）：\n\t- tokenizer.encode，其实就是把单词映射成对应的token id，结果是一维数组：{torch.tensor(ids).shape}\n\t- 经过embedding_layer后{emb.shape}\n\t- 经过squeeze 后：{result.shape}\n')
    return result

def cosine_sim(w1, w2):
    e1 = get_embedding(w1)
    e2 = get_embedding(w2)
    return F.cosine_similarity(e1.unsqueeze(0), e2.unsqueeze(0)).item()

for group_name, pairs in similarity_groups:
    print(f"--- {group_name} ---")
    for w1, w2 in pairs:
        sim = cosine_sim(w1, w2)
        bar = "█" * int((sim + 1) * 25)
        print(f"  cosine({w1:>4s}, {w2:>4s}) = {sim:+.4f}  {bar}")
    print()

print("💡 观察: 同类词(猫-狗, 苹果-香蕉)的相似度明显高于跨类词(猫-桌子, 苹果-汽车)")


=== 语义相似度对比 ===

--- 同类对比 ---
  cosine(king, queen) = +0.3574  █████████████████████████████████
  cosine(king,  man) = +0.0630  ██████████████████████████
  cosine(king, apple) = +0.1299  ████████████████████████████

--- 中文同类对比 ---
get_embedding 维度细节（输入词：猫）：
	- tokenizer.encode，其实就是把单词映射成对应的token id，结果是一维数组：torch.Size([1])
	- 经过embedding_layer后torch.Size([1, 1024])
	- 经过squeeze 后：torch.Size([1024])

  cosine(   猫,    狗) = +0.4727  ████████████████████████████████████
get_embedding 维度细节（输入词：猫）：
	- tokenizer.encode，其实就是把单词映射成对应的token id，结果是一维数组：torch.Size([1])
	- 经过embedding_layer后torch.Size([1, 1024])
	- 经过squeeze 后：torch.Size([1024])

  cosine(   猫,   桌子) = +0.0806  ███████████████████████████
  cosine(  苹果,   香蕉) = +0.3672  ██████████████████████████████████
  cosine(  苹果,   汽车) = +0.2109  ██████████████████████████████

💡 观察: 同类词(猫-狗, 苹果-香蕉)的相似度明显高于跨类词(猫-桌子, 苹果-汽车)


### 1.2 词类比：king - man + woman ≈ queen？

Word2Vec 时代最经典的发现：词向量之间存在**线性关系**。
如果 `king` 的向量减去 `man` 的向量再加上 `woman` 的向量，结果应该接近 `queen`。

但之前我们只用了**静态 Embedding 层**（模型的第一层），效果有限。
一个自然的问题：**经过完整的 Transformer 层之后，模型是否真正学到了这种语义关系？**

我们来对比静态 Embedding 和上下文 Embedding（跑完全部 Transformer 层后的输出）的词类比效果：

In [3]:
def get_contextual_embedding_from_sentence(sentence, target_word):
    """从句子中提取目标词的上下文 Embedding（经过全部 Transformer 层）"""
    inputs = tokenizer(sentence, return_tensors="pt")
    input_ids = inputs["input_ids"]
    token_ids_list = input_ids[0].tolist()
    decoded_tokens = [tokenizer.decode([tid]) for tid in token_ids_list]
    
    target_pos = None
    for j, dt in enumerate(decoded_tokens):
        if target_word in dt:
            target_pos = j
            break
    if target_pos is None:
        return None
    
    with torch.no_grad():
        outputs = model.model(input_ids)
        hidden_states = outputs.last_hidden_state
    return hidden_states[0, target_pos, :].detach()

print("=== 词类比实验: 静态 Embedding vs 上下文 Embedding ===\n")

analogies = [
    ("king", "man", "woman", "queen",
     "The king is here", "The man is here", "The woman is here", "The queen is here"),
    ("北京", "中国", "法国", "巴黎",
     "北京是城市", "中国是国家", "法国是国家", "巴黎是城市"),
    ("国王", "男人", "女人", "女王",
     "国王是人物", "男人是性别", "女人是性别", "女王是人物"),
]

all_ids = torch.arange(embedding_layer.num_embeddings)
with torch.no_grad():
    all_static_embeddings = embedding_layer(all_ids)

for a, b, c, expected, sa, sb, sc, sexpected in analogies:
    # --- 静态 Embedding ---
    ea_s = get_embedding(a)
    eb_s = get_embedding(b)
    ec_s = get_embedding(c)
    e_expected_s = get_embedding(expected)
    result_s = ea_s - eb_s + ec_s
    sim_s = F.cosine_similarity(result_s.unsqueeze(0), e_expected_s.unsqueeze(0)).item()
    
    sims_s = F.cosine_similarity(result_s.unsqueeze(0), all_static_embeddings)
    top5_s = torch.argsort(sims_s, descending=True)[:5]
    
    # --- 上下文 Embedding (经过全部 Transformer 层) ---
    ea_c = get_contextual_embedding_from_sentence(sa, a)
    eb_c = get_contextual_embedding_from_sentence(sb, b)
    ec_c = get_contextual_embedding_from_sentence(sc, c)
    e_expected_c = get_contextual_embedding_from_sentence(sexpected, expected)
    
    if all(x is not None for x in [ea_c, eb_c, ec_c, e_expected_c]):
        result_c = ea_c - eb_c + ec_c
        sim_c = F.cosine_similarity(result_c.unsqueeze(0), e_expected_c.unsqueeze(0)).item()
    else:
        sim_c = float("nan")
    
    print(f"  {a} - {b} + {c} ≈ {expected}?")
    print(f"  静态 Embedding:   与目标词相似度 = {sim_s:.4f}")
    print(f"    Top-5: ", end="")
    for tid in top5_s:
        tok_str = tokenizer.decode([tid.item()])
        s = sims_s[tid].item()
        marker = " <--" if tok_str.strip() == expected else ""
        print(f"{tok_str!r}({s:.3f}){marker}", end="  ")
    print()
    print(f"  上下文 Embedding: 与目标词相似度 = {sim_c:.4f}")
    print()

print("观察:")
print("  1. 静态 Embedding: queen 不在 king-man+woman 的 top-5 中，词类比几乎失败")
print("  2. 上下文 Embedding: 与目标词的相似度大幅提升（0.45 → 0.88），词类比明显更有效")
print("  3. 这说明 Transformer 层真正学到了语义关系，而不只是表面的向量距离")


=== 词类比实验: 静态 Embedding vs 上下文 Embedding ===

  king - man + woman ≈ queen?
  静态 Embedding:   与目标词相似度 = 0.4512
    Top-5: 'king'(0.719)  'キング'(0.465)  'вляю'(0.459)  'tifkan'(0.459)  ' поджелудо'(0.457)  
  上下文 Embedding: 与目标词相似度 = 0.8789

  北京 - 中国 + 法国 ≈ 巴黎?
  静态 Embedding:   与目标词相似度 = 0.4785
    Top-5: '法国'(0.617)  '北京'(0.555)  '法國'(0.520)  '巴黎'(0.479) <--  ' 프랑스'(0.439)  
  上下文 Embedding: 与目标词相似度 = 0.7812

  国王 - 男人 + 女人 ≈ 女王?
  静态 Embedding:   与目标词相似度 = 0.4258
    Top-5: '国王'(0.715)  '國王'(0.535)  '女王'(0.426) <--  '王妃'(0.393)  '女人'(0.389)  
  上下文 Embedding: 与目标词相似度 = 0.8789

观察:
  1. 静态 Embedding: queen 不在 king-man+woman 的 top-5 中，词类比几乎失败
  2. 上下文 Embedding: 与目标词的相似度大幅提升（0.45 → 0.88），词类比明显更有效
  3. 这说明 Transformer 层真正学到了语义关系，而不只是表面的向量距离


### 1.3 语境语义：同一个词，在不同语境下意思一样吗？

上面我们用的是**静态 Embedding**（Embedding 层的直接输出），每个 Token 只有一个固定的向量。
这意味着：无论"苹果"出现在"吃苹果"还是"苹果公司"中，静态 Embedding 给出的都是**同一个向量**。

但自然语言中，同一个词在不同语境下可能有完全不同的含义：
- "请帮我**开**门" → 开 = 打开
- "他**开**车上班" → 开 = 驾驶
- "这个**苹果**很甜" → 苹果 = 水果
- "**苹果**发布了新手机" → 苹果 = 公司

静态 Embedding 无法区分这些含义（同一个 Token 永远对应同一个向量，相似度 = 1.0）。
但经过 Transformer 层处理后，同一个 Token 会根据上下文获得不同的表示。
让我们对比一下：

In [4]:
def get_contextual_embedding(text, target_word):
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]
    token_ids_list = input_ids[0].tolist()
    decoded_tokens = [tokenizer.decode([tid]) for tid in token_ids_list]
    
    target_pos = None
    for i, dt in enumerate(decoded_tokens):
        if target_word in dt:
            target_pos = i
            break
    if target_pos is None:
        print(f"  Warning: target not found in: {decoded_tokens}")
        return None
    
    with torch.no_grad():
        outputs = model.model(input_ids)
        hidden_states = outputs.last_hidden_state
    return hidden_states[0, target_pos, :].detach()

print("=== 语境语义对比 ===\n")

contexts = [
    ("请帮我开门", "开", "打开"),
    ("他开车上班", "开", "驾驶"),
    ("这个苹果很甜", "苹果", "水果"),
    ("苹果发布了新手机", "苹果", "公司"),
    ("今天天气很热", "热", "温度高"),
    ("他是个热门人物", "热", "受欢迎"),
    ("这朵花很美", "花", "花朵"),
    ("他花了很多钱", "花", "花费"),
]

emb_dict = {}
for text, word, sense in contexts:
    emb = get_contextual_embedding(text, word)
    if emb is not None:
        key = f"{word}({sense})"
        emb_dict[key] = emb
        print(f'  "{text}" -> {key}')

print(f"\n--- 核心对比: 静态 Embedding vs 上下文 Embedding ---")
print(f"静态 Embedding: 同一个 Token 永远是同一个向量，所以同词不同义的相似度 = 1.0")
print(f"上下文 Embedding: 经过 Transformer 层后，同词不同义的相似度 < 1.0\n")

word_groups = [
    ("开", ["打开", "驾驶"]),
    ("热", ["温度高", "受欢迎"]),
    ("苹果", ["水果", "公司"]),
    ("花", ["花朵", "花费"]),
]

header = f'  {"词":<6s} {"含义1":<8s} {"含义2":<8s} {"静态相似度":<12s} {"上下文相似度":<12s} {"差异":<8s}'
print(header)
print("  " + "-" * (len(header) - 2))

for word, senses in word_groups:
    keys = [f"{word}({s})" for s in senses]
    valid_keys = [k for k in keys if k in emb_dict]
    if len(valid_keys) >= 2:
        ctx_sim = F.cosine_similarity(emb_dict[valid_keys[0]].unsqueeze(0), emb_dict[valid_keys[1]].unsqueeze(0)).item()
        diff = 1.0 - ctx_sim
        print(f"  {word:<6s} {senses[0]:<8s} {senses[1]:<8s} {1.0:<12.4f} {ctx_sim:<12.4f} {diff:<8.4f}")

print(f"\n--- 更有意思的发现 ---")
if "花(花朵)" in emb_dict and "苹果(水果)" in emb_dict and "花(花费)" in emb_dict:
    sim_cross = F.cosine_similarity(emb_dict["花(花朵)"].unsqueeze(0), emb_dict["苹果(水果)"].unsqueeze(0)).item()
    sim_same = F.cosine_similarity(emb_dict["花(花朵)"].unsqueeze(0), emb_dict["花(花费)"].unsqueeze(0)).item()
    print(f'  "花(花朵)" vs "苹果(水果)" = {sim_cross:.4f}  (不同词，同类含义)')
    print(f'  "花(花朵)" vs "花(花费)" = {sim_same:.4f}  (同一个词，不同含义)')
    if sim_cross > sim_same:
        print(f"  -> 模型认为'花(花朵)'跟'苹果(水果)'比跟'花(花费)'更近！")
        print(f"     这说明 Transformer 真正理解了语义，而不是仅仅看字面相同")

print(f"\n观察:")
print(f"  1. 静态 Embedding: 同词不同义相似度 = 1.0（完全无法区分）")
print(f"  2. 上下文 Embedding: 同词不同义相似度 < 1.0（Transformer 能区分不同含义）")
print(f"  3. '开(打开)' vs '开(驾驶)' 差异最大(0.53)，因为这两个含义几乎毫无关系")
print(f"  4. 语义相近的词（花-苹果）甚至比字面相同的词（花-花）更接近！")


=== 语境语义对比 ===

  "请帮我开门" -> 开(打开)
  "他开车上班" -> 开(驾驶)
  "这个苹果很甜" -> 苹果(水果)
  "苹果发布了新手机" -> 苹果(公司)
  "今天天气很热" -> 热(温度高)
  "他是个热门人物" -> 热(受欢迎)
  "这朵花很美" -> 花(花朵)
  "他花了很多钱" -> 花(花费)

--- 核心对比: 静态 Embedding vs 上下文 Embedding ---
静态 Embedding: 同一个 Token 永远是同一个向量，所以同词不同义的相似度 = 1.0
上下文 Embedding: 经过 Transformer 层后，同词不同义的相似度 < 1.0

  词      含义1      含义2      静态相似度        上下文相似度       差异      
  -----------------------------------------------------------
  开      打开       驾驶       1.0000       0.4707       0.5293  
  热      温度高      受欢迎      1.0000       0.5508       0.4492  
  苹果     水果       公司       1.0000       0.6797       0.3203  
  花      花朵       花费       1.0000       0.7383       0.2617  

--- 更有意思的发现 ---
  "花(花朵)" vs "苹果(水果)" = 0.7891  (不同词，同类含义)
  "花(花朵)" vs "花(花费)" = 0.7383  (同一个词，不同含义)
  -> 模型认为'花(花朵)'跟'苹果(水果)'比跟'花(花费)'更近！
     这说明 Transformer 真正理解了语义，而不是仅仅看字面相同

观察:
  1. 静态 Embedding: 同词不同义相似度 = 1.0（完全无法区分）
  2. 上下文 Embedding: 同词不同义相似度 < 1.0（Transformer 能区分不同含义）
  3. '开(打开)' vs '开

---

## 小结

本节我们验证了 Embedding 的三大特性：

1. **语义相似度**：同类词（猫-狗）的向量比跨类词（猫-桌子）更接近
2. **词类比**：king - man + woman ≈ queen，上下文 Embedding 比静态 Embedding 效果好得多
3. **语境语义**：同一个词在不同语境下获得不同的向量表示，Transformer 真正理解了语义

下一步：有了可计算的向量表示，模型如何从中预测出下一个 Token？👉 [01-3 Token 生 Token 之愚公移山](./01-3-token_generates_token.ipynb)